In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import glob
import os
from sklearn.model_selection import GroupKFold

base_path = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'

print("1. 基準井戸(Typewell)データを読み込んで辞書にまとめています...")
# trainとtest両方のフォルダからTypewellを探して読み込む
typewell_files = glob.glob(f'{base_path}/**/*__typewell.csv', recursive=True)
typewells = {}
for f in typewell_files:
    well_id = os.path.basename(f).split('__')[0]
    # 後で深さ(TVT)で検索しやすいようにソートしておく
    df = pd.read_csv(f).sort_values('TVT')
    typewells[well_id] = df

# -------------------------------------------------------------
# ★ メダルに向けた新兵器：Typewellの情報を紐づける関数
# -------------------------------------------------------------
def add_typewell_features(horiz_df, well_id):
    if well_id not in typewells:
        horiz_df['ref_GR'] = horiz_df['GR'] # 万が一Typewellが無い場合の保険
        return horiz_df
    
    # 基準井戸のTVTとGRだけを取り出し、名前を変える
    t_df = typewells[well_id][['TVT', 'GR']].copy()
    t_df.columns = ['ref_TVT', 'ref_GR'] 
    
    # 水平井の元の行の順番を記録しておく
    horiz_df['orig_idx'] = np.arange(len(horiz_df))
    
    # ドリルの深さ(Z)を基準にして、一番近い深さ(ref_TVT)の設計図データをくっつける
    h_sorted = horiz_df.sort_values('Z')
    merged = pd.merge_asof(
        h_sorted,
        t_df,
        left_on='Z',
        right_on='ref_TVT',
        direction='nearest'
    )
    
    # 元の行の順番に戻して、作業用の列を捨てる
    merged = merged.sort_values('orig_idx').drop(columns=['orig_idx', 'ref_TVT'])
    return merged
# -------------------------------------------------------------

print("2. 学習データ(train)を読み込み、Typewellと結合しています...")
train_files = glob.glob(f'{base_path}/train/*__horizontal_well.csv')
train_list = []
for f in train_files:
    well_id = os.path.basename(f).split('__')[0]
    df = pd.read_csv(f)
    df['well_id'] = well_id
    df = add_typewell_features(df, well_id) # ★ここでカンペを追加！
    train_list.append(df)
train_df = pd.concat(train_list, ignore_index=True)

print("3. テストデータ(test)を読み込み、Typewellと結合しています...")
test_files = glob.glob(f'{base_path}/test/*__horizontal_well.csv')
test_list = []
for f in test_files:
    well_id = os.path.basename(f).split('__')[0]
    df = pd.read_csv(f)
    df['well_id'] = well_id
    df['id'] = well_id + '_' + df.index.astype(str)
    df = add_typewell_features(df, well_id) # ★ここでもカンペを追加！
    test_list.append(df)
test_df = pd.concat(test_list, ignore_index=True)

print("4. モデルの学習を開始します...")
# ★ 特徴量に 'ref_GR'（設計図のガンマ線）を追加！
features = ['MD', 'X', 'Y', 'Z', 'GR', 'ref_GR']
target = 'TVT'

# 欠損値の処理と削除
train_df[features] = train_df[features].fillna(0)
test_df[features] = test_df[features].fillna(0)
train_df = train_df.dropna(subset=[target])

# GroupKFoldによるモデル学習
gkf = GroupKFold(n_splits=5)
models = []
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, train_df[target], train_df['well_id'])):
    X_tr, y_tr = train_df.iloc[train_idx][features], train_df.iloc[train_idx][target]
    X_va, y_va = train_df.iloc[val_idx][features], train_df.iloc[val_idx][target]
    
    model = lgb.LGBMRegressor(n_estimators=100, random_state=42)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
    models.append(model)

print("5. 予測と提出ファイルの作成を行っています...")
preds = np.zeros(len(test_df))
for model in models:
    preds += model.predict(test_df[features]) / len(models)

test_df['predicted_tvt'] = preds

sub = pd.read_csv(f'{base_path}/sample_submission.csv')
sub = sub.drop(columns=['tvt']).merge(test_df[['id', 'predicted_tvt']], on='id', how='left')
sub = sub.rename(columns={'predicted_tvt': 'tvt'})
sub['tvt'] = sub['tvt'].fillna(0.0)

sub[['id', 'tvt']].to_csv('submission.csv', index=False)
print("完了しました！submission.csv が作成されました。")